In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

import scipy.optimize

# Needed for animations
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

Based on "Thermal activation of metastable decay: Testing nucleation theory" by Mark Alford, Hume Feldman and Marcelo Gleiser. *Physical Review D* **47** R2168 (1993).

https://doi.org/10.1103/PhysRevD.47.R2168

This paper did nucleation in 1+1 dimensions, and there have been plenty of works in both 1+1 and higher dimensions since then, but it still gives the basic idea.

**NOTE** The paper uses two variants of epsilon, one (𝜀) for the timestep and another (𝜖) for the cubic term in the potential. To avoid confusion we call the first one 'dt'.

In [ ]:
@dataclass
class Parameters:
    # Number of lattice sites
    Nx: int
    # Lattice spacing, called l in this paper, but more usually a or dx
    l: float
    # Strength of the cubic coupling
    epsilon: float
    # Time step, labelled with a variant of epsilon in this paper, but more usually called dt
    # - to avoid confusion we call it dt
    dt: float
    # The rescaled viscosity, the damping term in the equations of motion
    etatilde: float
    # The temeperature of the thermal bath
    theta: float

    # Random number generator, needs initialised with a suitable seed
    rng: np.random.Generator

Remember, we work with phi0 = 1.

In [ ]:
# The rescaled potential, eq. 1 with epsilon substituted for alpha.
def potential(phi: float, p: Parameters):
    return (1.0 / 4.0) * phi**2 * (phi - 2) ** 2 - p.epsilon * phi**3


# dV/dphi for the above potential
def grad_potential(phi: float, p: Parameters):
    return phi * (phi - 1) * (phi - 2) - 3 * p.epsilon * phi**2


def extrema(p: Parameters):
    return np.roots([1, -3 - 3 * p.epsilon, 2, 0])


# The 'turning point' i.e. saddle point of the bounce action (not the potential)
def get_phi_tp(p: Parameters):
    phi_tp = 2.0 * ((1 + p.epsilon) - (2 * p.epsilon + p.epsilon**2) ** 0.5)
    return phi_tp


# The scale of the noise (given below eq 10)
def get_noise_scale(p: Parameters):
    return (2 * p.etatilde * p.theta / (p.l * p.epsilon)) ** 0.5

In [ ]:
# Equation 10, the leapfrog equations of motion


def step_pi(phi: np.ndarray, pi: np.ndarray, p: Parameters):
    noise_scale = get_noise_scale(p)

    for n in range(p.Nx):
        laplacian = phi[(n - 1 + p.Nx) % p.Nx] + phi[(n + 1) % p.Nx] - 2.0 * phi[n]

        dVdPhi = grad_potential(phi[n], p)
        noise = p.rng.normal(scale=noise_scale)
        pi[n] = pi[n] + p.dt * (
            laplacian / p.l**2 - p.etatilde * pi[n] - dVdPhi + noise
        )

    return pi


def step_phi(phi: np.ndarray, pi: np.ndarray, p: Parameters):
    phi = phi + p.dt * pi

    return phi


def run(phi, pi, steps):
    for i in range(steps):
        pi = step_pi(phi, pi, p)
        phi = step_phi(phi, pi, p)

    return phi, pi

In [ ]:
p = Parameters(
    Nx=200,
    l=1.0,
    epsilon=0.1,
    dt=0.1,
    etatilde=0.1,
    theta=0.2,
    rng=np.random.default_rng(),
)

x = np.arange(-1.4, 3.6, 0.01)
plt.xlabel(r"$\phi$")
plt.ylabel(r"$V(\phi)$")
plt.plot(x, potential(x, p), label="Potential")
tp = get_phi_tp(p)
plt.scatter(tp, potential(tp, p), color="black", label="Turning point")
ext = extrema(p)
plt.scatter(ext, potential(ext, p), color="orange", label="Extrema")

plt.legend()

Run the simulation briefly with the above parameters, starting from a cold initial configuraration.

In [ ]:
phi = np.zeros(p.Nx)
pi = np.zeros(p.Nx)

phi_snapshots = []

for i in range(400):
    phi, pi = run(phi, pi, 2)
    phi_snapshots.append(phi)

Plot everything as a little animation:

In [ ]:
def plot_slices():
    fig, ax = plt.subplots()

    def update(frame):
        ax.clear()
        state_plot = ax.plot(range(p.Nx), phi_snapshots[frame])
        for val in ext:
            ax.plot((0, p.Nx), (val, val), color="orange", ls="--")
        ax.plot((0, p.Nx), (tp, tp), color="black", ls="--")
        ax.set_xlabel(r"$x$")
        ax.set_ylim(-2, 4)
        ax.set_ylabel(r"$\phi(x)$")

    ani = FuncAnimation(
        fig, update, frames=range(0, len(phi_snapshots), 1), interval=100, repeat=False
    )
    return ani


movie = plot_slices().to_jshtml()
plt.close()
HTML(movie)

Now that we can see stuff nucleate, let's try to do something more scientific and build the blocks we need to start reproducing the results of the original paper (making more progress is left as an exercise to you!)...

First, let's define the 'naive' smoothing algorithm they use.

In [ ]:
# Smooth the field by averaging over a shifting window of deltaL sites.
def smooth(phi: np.ndarray, deltaL: int, p: Parameters):
    phi_smoothed = np.zeros_like(phi)

    for n in range(p.Nx):
        count = 0
        # '//' means integer division
        for delta in range(-(deltaL // 2), deltaL // 2):
            count += 1
            phi_smoothed[n] += phi[(n + delta + p.Nx) % p.Nx]
        phi_smoothed[n] /= count
        # print(phi_smoothed[n], count)

    return phi_smoothed

Now the code to run a simulation until nucleation occurs, according to their basic definition.

In [ ]:
# Run a simulation for up to max_steps, or until it nucleates, whichever comes first.
# Returns the time it took to pass the nucleation criterion, or None if it never does.


def run_until_nucleated(phi, pi, max_steps):
    phi_tp = get_phi_tp(p)
    time = -0.5 * p.epsilon

    for i in range(max_steps):
        time += p.epsilon
        pi = step_pi(phi, pi, p)
        phi = step_phi(phi, pi, p)

        phi_smoothed = smooth(phi, 20, p)
        if np.max(phi_smoothed) > phi_tp:
            return time

    return None

Note that `run_until_nucleated()` takes in a number of steps but returns a dimensionless physical time.

In [ ]:
phi = np.zeros(p.Nx)
pi = np.zeros(p.Nx)
run_until_nucleated(phi, pi, 1000)

Now build code to get the distribution of nucleation times given different random seeds. This code reruns the same simulation.

Note that, per the documentation, https://numpy.org/doc/stable/reference/random/generator.html - if no seed is passed to the random number generator "fresh, unpredictable entropy will be pulled from the OS". So each rerun should be independent.

In [ ]:
def get_nucleation_times(
    p: Parameters, reruns=100, max_steps=1000, show_progress=False
):

    times = []

    for i in range(reruns):
        # Fresh random number generator, overriding what we were given
        p.rng = np.random.default_rng()

        if show_progress:
            print("Rerun %d/%d" % (i, reruns))

        # Cold start
        phi = np.zeros(p.Nx)
        pi = np.zeros(p.Nx)

        this_time = run_until_nucleated(phi, phi, max_steps)
        times.append(this_time)

    return times

This takes about 14 seconds on my laptop (MacBook Air M4):

In [ ]:
times = np.array(get_nucleation_times(p))

In [ ]:
if None in times:
    print("WARNING: not all reruns nucleated")

In [ ]:
plt.hist(times, bins=20)
plt.plot()

Survival time plot:

In [ ]:
survival_time = np.arange(0, 100)
survival_fraction = np.array(
    [np.count_nonzero(times > t) for t in survival_time]
) / np.size(times)
plt.semilogy(survival_time, survival_fraction)
plt.xlim(p.epsilon, max(times))

## Work in progress beyond this point!

Reproduce (part of) Fig. 1 in the paper.

In [ ]:
epsilon = 0.1
thetas = [1.0 / 8.0, 1.0 / 10.0, 1.0 / 12.0, 1.0 / 14.0]
max_steps = 1e6
max_time = max_steps * p.dt
survival_time = np.arange(0, max_time)
mean_time = []
for theta in thetas:
    p.theta = theta
    print("Doing theta=%g" % theta)
    times = np.array(get_nucleation_times(p, reruns=20, max_steps=10000000))
    if None in times:
        print("ERROR: One of the runs did not nucleate, stopping")
        break
    survival_fraction = np.array(
        [np.count_nonzero(times > t) for t in survival_time]
    ) / np.size(times)

    mean_time.append(np.where(survival_fraction < 0.5)[0][0])

In [ ]:
mean_time

In [ ]:
plt.semilogy(1.0 / np.array(thetas), mean_time)